In [8]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("northmart_dev.silver.fraud_transactions_silver")

features_300min = (
    df
    .groupBy(
        "card_id",
        F.window("event_time", "300 minutes")
    )
    .agg(
        F.count("*").alias("tx_count_300min"),
        F.sum("amount").alias("amount_sum_300min")
    )
)

features_24h = (
    df
    .groupBy(
        "card_id",
        F.window("event_time", "24 hours")
    )
    .agg(
        F.avg("amount").alias("avg_amount_24h")
    )
)

display(features_30min.limit(20))
display(features_24h.limit(20))

,card_id,window,tx_count_30min,amount_sum_30min
0,CARD-007008,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,91.36
1,CARD-004002,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",2,2162.75
2,CARD-004054,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,75.87
3,CARD-003136,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,241.03
4,CARD-014966,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,161.92
5,CARD-007223,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,267.70
6,CARD-010498,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,229.39
7,CARD-008077,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,134.01
8,CARD-012000,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,115.38
9,CARD-008719,"{'start': 2026-08-21 11:00:00, 'end': 2026-08-21 16:00:00}",1,258.49


,card_id,window,avg_amount_24h
0,CARD-007008,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",91.3600
1,CARD-004002,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",1081.3750
2,CARD-004054,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",75.8700
3,CARD-003136,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",241.0300
4,CARD-014966,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",161.9200
5,CARD-007223,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",267.7000
6,CARD-010498,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",229.3900
7,CARD-008077,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",134.0100
8,CARD-012000,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",115.3800
9,CARD-008719,"{'start': 2026-08-21 00:00:00, 'end': 2026-08-22 00:00:00}",258.4900


In [13]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

transactions = spark.table(
    "northmart_dev.silver.fraud_transactions_silver"
)

df = transactions.withColumn(
    "event_time_sec",
    F.col("event_time").cast("long")
)

w_300min = (
    Window
    .partitionBy("card_id")
    .orderBy("event_time_sec")
    .rangeBetween(-300 * 60, -1)
)

features_test = (
    df
    .withColumn(
        "tx_count_300min",
        F.count("*").over(w_300min)
    )
    .withColumn(
        "amount_sum_300min",
        F.sum("amount").over(w_300min)
    )
)

w_24h = (
    Window
    .partitionBy("card_id")
    .orderBy("event_time_sec")
    .rangeBetween(-24 * 60 * 60, -1)
)

features_test = (
    features_test
    .withColumn(
        "avg_amount_24h",
        F.avg("amount").over(w_24h)
    )
)

features_test = features_test.withColumn(
    "amount_ratio_vs_24h_avg",
    F.when(
        F.col("avg_amount_24h") > 0,
        F.col("amount") / F.col("avg_amount_24h")
    )
)

display(
    features_test
    .select(
        "card_id",
        "event_time",
        "event_time_sec",
        "amount",
        "tx_count_300min",
        "amount_sum_300min",
        "avg_amount_24h",
        "amount_ratio_vs_24h_avg"
    )
    .orderBy(F.desc("tx_count_300min"))
    .limit(30)
)

,card_id,event_time,event_time_sec,amount,tx_count_300min,amount_sum_300min,avg_amount_24h,amount_ratio_vs_24h_avg
0,CARD-004158,2026-08-21 16:07:45.745737,1787328465,51.96,3,7353.41,2451.136667,0.021198
1,CARD-009628,2026-08-21 15:22:20.285349,1787325740,225.11,2,337.57,168.785000,1.333709
2,CARD-000405,2026-08-21 15:50:51.414533,1787327451,128.98,2,458.94,229.470000,0.562078
3,CARD-011048,2026-08-21 15:51:53.495839,1787327513,23.99,2,488.31,244.155000,0.098257
4,CARD-002318,2026-08-21 16:05:18.553321,1787328318,26.41,2,251.08,125.540000,0.210371
5,CARD-007595,2026-08-21 16:09:50.911244,1787328590,86.59,2,183.49,91.745000,0.943812
6,CARD-001082,2026-08-21 16:01:38.266460,1787328098,268.58,2,3306.54,1653.270000,0.162454
7,CARD-002205,2026-08-21 15:58:09.995433,1787327889,248.23,2,503.87,251.935000,0.985294
8,CARD-010396,2026-08-21 16:10:25.958165,1787328625,258.00,2,253.31,126.655000,2.037030
9,CARD-002932,2026-08-21 16:10:38.975401,1787328638,260.88,2,415.07,207.535000,1.257041


In [ ]:
from databricks.feature_engineering import (
    FeatureEngineeringClient,
    FeatureLookup
)

fe = FeatureEngineeringClient()

transactions = spark.table(
    "northmart_dev.silver.fraud_transactions_silver"
)

feature_lookups = [
    FeatureLookup(
        table_name="northmart_dev.ml.fraud_features",
        lookup_key="card_id",
        timestamp_lookup_key="event_time",
        feature_names=[
            "transaction_count_5min",
            "amount_sum_5min"
        ]
    )
]

training_set = fe.create_training_set(
    df=transactions.select(
        "transaction_id",
        "card_id",
        "event_time",
        "is_fraud"
    ),
    feature_lookups=feature_lookups,
    label="is_fraud",
    exclude_columns=["transaction_id"]
)

In [4]:
import mlflow
import mlflow.sklearn

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run() as run:
    fe.log_model(
        model=model,
        artifact_path="fraud_model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name=None
    )

    mlflow.log_metric("roc_auc_train", roc_auc_score(y, y_proba))

    print("Run ID:", run.info.run_id)

/Users/gerald/northmart360/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(
{"ts": "2026-08-23 22:28:44.325", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session has changed from cct.CklzcGFya2Nvbm5lY3Qvc2Vzc2lvbi8xNDc3MTg5MTAwNDk5MDkvNzYwMTFhNWEtZGZmZC00NTZlLTg1NTEtMGY0ZDJkNDEwNmY5EAEgAjIkMDFhMDMwMzEtYmFjNC03YjMwLThhODAtODJlMGNhMjM5NDFjOiRlMTFjODI1NS1kMGE5LTM3OGMtYWU4Zi04MzgyZjM4YjdiOGFKDAjQoK3UBhCAssiUAlABWAFgAWiNqrrC2+uTDQ== to cct.CklzcGFya2Nvbm5lY3Qvc2Vzc2lvbi8xNDc3MTg5MTAwNDk5MDkvNzYwMTFhNWEtZGZmZC00NTZlLTg1NTEtMGY0ZDJkNDEwNmY5EAEgAjIkMDFhMDMwNGYtODg0OS03MGU0LWJiOWYtYmI3M2RiZGYwMmZlOiRlMTFjODI1NS1kMGE5LTM3OGMtYWU4Zi04MzgyZjM4YjdiOGFKDAjxr63UBhDA9LbfAlABWAFgAWiNqr

NameError: name 'model' is not defined